In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

data = {
    "Student Name": [
        "Arun",
        "Bala",
        "Charan",
        "David",
        "Ezhil",
        "Ganesh",
        "Hari",
        "Karthik",
        "Manoj",
        "Naveen",
        "Praveen",
        "Ravi",
    ],
    "Study Hours": [2.5, 5.1, 8.2, 1.5, 7.0, 9.5, 2.0, 6.5, 3.0, 8.0, 7.5, 1.8],
    "Attendance": [75, 85, 90, 60, 95, 98, 70, 88, 65, 92, 90, 62],
    "Previous Marks": [50, 65, 80, 40, 75, 85, 45, 78, 48, 82, 80, 42],
    "Final Marks": [48, 68, 85, 38, 78, 92, 42, 80, 45, 86, 84, 40],
    "Result": [
        "Fail",
        "Pass",
        "Pass",
        "Fail",
        "Pass",
        "Pass",
        "Fail",
        "Pass",
        "Fail",
        "Pass",
        "Pass",
        "Fail",
    ],
}

df = pd.DataFrame(data)
print("STUDENT DATASET")
print(df.to_string(index=False))
print("\nDataset Information:")
df.info()
print("\nMissing Values:")
print(df.isnull().sum())
print("\nStatistical Summary:")
print(df.describe())
print("\nDataset Shape:")
print(df.shape)

# Regression Model
X = df[["Study Hours", "Attendance", "Previous Marks"]]
y = df["Final Marks"]

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X, y, test_size=0.25, random_state=42
)

reg_model = LinearRegression()
reg_model.fit(X_train_reg, y_train_reg)
reg_predictions = reg_model.predict(X_test_reg)

print("\nActual Marks:")
print(y_test_reg.values)
print("\nPredicted Marks:")
print(np.round(reg_predictions, 2))

mae = mean_absolute_error(y_test_reg, reg_predictions)
mse = mean_squared_error(y_test_reg, reg_predictions)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_reg, reg_predictions)

print("\nREGRESSION EVALUATION")
print("MAE:", round(mae, 2))
print("MSE:", round(mse, 2))
print("RMSE:", round(rmse, 2))
print("R2 Score:", round(r2, 2))

# Classification Model
X = df[["Study Hours", "Attendance", "Previous Marks"]]
y_class = df["Result"].map({"Fail": 0, "Pass": 1})

X_train_class, X_test_class, y_train_class, y_test_class = train_test_split(
    X, y_class, test_size=0.25, random_state=42, stratify=y_class
)

class_model = make_pipeline(
    StandardScaler(), LogisticRegression(max_iter=2000)
)
class_model.fit(X_train_class, y_train_class)
class_predictions = class_model.predict(X_test_class)

print("\nActual Results:")
print(y_test_class.values)
print("\nPredicted Results:")
print(["Pass" if value == 1 else "Fail" for value in class_predictions])

accuracy = accuracy_score(y_test_class, class_predictions)
precision = precision_score(y_test_class, class_predictions, zero_division=0)
recall = recall_score(y_test_class, class_predictions, zero_division=0)
f1 = f1_score(y_test_class, class_predictions, zero_division=0)

print("\nCLASSIFICATION EVALUATION")
print("Accuracy:", round(accuracy, 2))
print("Precision:", round(precision, 2))
print("Recall:", round(recall, 2))
print("F1 Score:", round(f1, 2))

cm = confusion_matrix(y_test_class, class_predictions, labels=[0, 1])
print("\nConfusion Matrix:")
print(cm)

# Enter Multiple Student Details One by One & Append to Existing Dataset
print("\n====================================")
print("       ENTER STUDENT DETAILS        ")
print("====================================")

num_students = int(input("How many student records do you want to input? "))
if num_students <= 0:
  raise ValueError("Number of students must be greater than 0.")

new_records = []

for i in range(1, num_students + 1):
  print(f"\n--- Entering Details for Student {i} of {num_students} ---")
  student_name = input("Enter Student Name: ")
  study_hours = float(input("Enter Study Hours: "))
  attendance = float(input("Enter Attendance Percentage: "))
  previous_marks = float(input("Enter Previous Marks: "))

  if not student_name.strip():
    raise ValueError("Student name cannot be empty.")
  if study_hours < 0:
    raise ValueError("Study hours cannot be negative.")
  if not 0 <= attendance <= 100:
    raise ValueError("Attendance must be between 0 and 100.")
  if not 0 <= previous_marks <= 100:
    raise ValueError("Previous marks must be between 0 and 100.")

  new_student = pd.DataFrame({
      "Study Hours": [study_hours],
      "Attendance": [attendance],
      "Previous Marks": [previous_marks],
  })

  predicted_marks = reg_model.predict(new_student)
  predicted_marks_clipped = round(float(np.clip(predicted_marks, 0, 100)[0]), 2)

  predicted_result = class_model.predict(new_student)
  final_result = "Pass" if predicted_result[0] == 1 else "Fail"

  print("\n====================================")
  print("           STUDENT REPORT           ")
  print("====================================")
  print("Student Name:", student_name.strip())
  print("Study Hours:", study_hours)
  print("Attendance:", attendance, "%")
  print("Previous Marks:", previous_marks)
  print("------------------------------------")
  print("Predicted Final Marks:", predicted_marks_clipped)
  print("Predicted Result:", final_result)
  print("====================================")

  new_records.append({
      "Student Name": student_name.strip(),
      "Study Hours": study_hours,
      "Attendance": attendance,
      "Previous Marks": previous_marks,
      "Final Marks": predicted_marks_clipped,
      "Result": final_result,
  })

# Append all newly entered student records to the existing dataset
df = pd.concat([df, pd.DataFrame(new_records)], ignore_index=True)

print("\n====================================")
print("      UPDATED STUDENT DATASET       ")
print("====================================")
print("Updated Dataset Shape:", df.shape)
print("\nNewly Added Records:")
print(df.tail(num_students).to_string(index=False))
print("\nComplete Updated Dataset:")
print(df.to_string(index=False))

comparison = pd.DataFrame({
    "Comparison": [
        "Algorithm Used",
        "Target Variable",
        "Type of Prediction",
        "Evaluation Metrics",
        "Real-World Application",
    ],
    "Regression Model": [
        "Linear Regression",
        "Final Marks",
        "Numerical Prediction",
        "MAE, MSE, RMSE, R2 Score",
        "Predicting Student Marks",
    ],
    "Classification Model": [
        "Logistic Regression",
        "Pass or Fail",
        "Categorical Prediction",
        "Accuracy, Precision, Recall, F1 Score",
        "Predicting Student Results",
    ],
})

print("\nCOMPARISON OF BOTH MODELS")
print(comparison.to_string(index=False))

STUDENT DATASET
Student Name  Study Hours  Attendance  Previous Marks  Final Marks Result
        Arun          2.5          75              50           48   Fail
        Bala          5.1          85              65           68   Pass
      Charan          8.2          90              80           85   Pass
       David          1.5          60              40           38   Fail
       Ezhil          7.0          95              75           78   Pass
      Ganesh          9.5          98              85           92   Pass
        Hari          2.0          70              45           42   Fail
     Karthik          6.5          88              78           80   Pass
       Manoj          3.0          65              48           45   Fail
      Naveen          8.0          92              82           86   Pass
     Praveen          7.5          90              80           84   Pass
        Ravi          1.8          62              42           40   Fail

Dataset Information:


How many student records do you want to input?  1



--- Entering Details for Student 1 of 1 ---


Enter Student Name:  hari
Enter Study Hours:  4
Enter Attendance Percentage:  85
Enter Previous Marks:  77



           STUDENT REPORT           
Student Name: hari
Study Hours: 4.0
Attendance: 85.0 %
Previous Marks: 77.0
------------------------------------
Predicted Final Marks: 75.58
Predicted Result: Pass

      UPDATED STUDENT DATASET       
Updated Dataset Shape: (13, 6)

Newly Added Records:
Student Name  Study Hours  Attendance  Previous Marks  Final Marks Result
        hari          4.0        85.0            77.0        75.58   Pass

Complete Updated Dataset:
Student Name  Study Hours  Attendance  Previous Marks  Final Marks Result
        Arun          2.5        75.0            50.0        48.00   Fail
        Bala          5.1        85.0            65.0        68.00   Pass
      Charan          8.2        90.0            80.0        85.00   Pass
       David          1.5        60.0            40.0        38.00   Fail
       Ezhil          7.0        95.0            75.0        78.00   Pass
      Ganesh          9.5        98.0            85.0        92.00   Pass
        Hari 